# VAI NVS Competition — 3D Gaussian Splatting Pipeline

**Yêu cầu Kaggle:**
- GPU: T4 x2 (hoặc P100)
- Internet: **BẬT** khi chạy Cell 1-2, có thể tắt sau
- Dataset: Upload `phase1/` lên Kaggle Dataset tên `vai-nvs-phase1`

**Ước tính thời gian:** ~9-10 giờ cho 13 scenes (trong giới hạn 12h session)

In [ ]:
# ============================================================
# CELL 1: Kiểm tra GPU & Disk
# ============================================================
import subprocess, os, shutil, time

print('=== GPU INFO ===')
subprocess.run(['nvidia-smi'], check=True)

print('\n=== DISK SPACE ===')
total, used, free = shutil.disk_usage('/kaggle/working')
print(f'Free: {free/1e9:.1f} GB / Total: {total/1e9:.1f} GB')

print('\n=== TORCH CUDA ===')
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

In [ ]:
# ============================================================
# CELL 2: Clone & Compile Gaussian Splatting
# (Internet phải BẬT — chạy 1 lần, mất ~10-15 phút)
# ============================================================
import subprocess, os

GS_DIR = '/kaggle/working/gaussian-splatting'

if not os.path.exists(GS_DIR):
    print('Cloning gaussian-splatting...')
    subprocess.run([
        'git', 'clone', '--recursive',
        'https://github.com/graphdeco-inria/gaussian-splatting',
        GS_DIR
    ], check=True)
else:
    print('gaussian-splatting already cloned.')

print('\nInstalling Python dependencies...')
subprocess.run(['pip', 'install', '-q', 'plyfile', 'tqdm', 'lpips', 'scikit-image'], check=True)

print('\nCompiling diff-gaussian-rasterization (mất ~5-8 phút)...')
subprocess.run(
    ['pip', 'install', '-q', f'{GS_DIR}/submodules/diff-gaussian-rasterization'],
    check=True
)

print('Compiling simple-knn...')
subprocess.run(
    ['pip', 'install', '-q', f'{GS_DIR}/submodules/simple-knn'],
    check=True
)

print('\n✅ Setup complete!')

In [ ]:
# ============================================================
# CELL 2.5: Apply Custom Patches to Gaussian Splatting Repo
# ============================================================
import subprocess, os, re

GS_DIR = '/kaggle/working/gaussian-splatting'

# Reset files to clean git state first to prevent corruption from multiple runs
print('Resetting modified files to clean git state...')
subprocess.run(['git', 'checkout', 'scene/dataset_readers.py'], cwd=GS_DIR)
subprocess.run(['git', 'checkout', 'utils/camera_utils.py'], cwd=GS_DIR)
subprocess.run(['git', 'checkout', 'scene/gaussian_model.py'], cwd=GS_DIR)

# == Patch 1: dataset_readers.py ==
dr_path = f'{GS_DIR}/scene/dataset_readers.py'
with open(dr_path) as f:
    code = f.read().replace('\r\n', '\n')

# 1a: Add SIMPLE_RADIAL support
OLD1 = '''        else:
            assert False, "Colmap camera model not handled: only undistorted datasets (PINHOLE or SIMPLE_PINHOLE cameras) supported!"'''
OLD1 = OLD1.replace('\r\n', '\n')

NEW1 = '''        elif intr.model in ("SIMPLE_RADIAL", "RADIAL"):
            focal_length_x = intr.params[0]
            FovY = focal2fov(focal_length_x, height)
            FovX = focal2fov(focal_length_x, width)
        elif intr.model == "OPENCV":
            focal_length_x = intr.params[0]
            focal_length_y = intr.params[1]
            FovY = focal2fov(focal_length_y, height)
            FovX = focal2fov(focal_length_x, width)
        else:
            raise ValueError(f"Unsupported camera model: {intr.model}")'''
NEW1 = NEW1.replace('\r\n', '\n')

if OLD1 in code:
    code = code.replace(OLD1, NEW1)
    print('Patch 1a applied: SIMPLE_RADIAL support')
else:
    print('Patch 1a: pattern not found, trying regex...')
    code = re.sub(
        r'else:\\s*\\n\\s*assert False, "Colmap camera model not handled.*?"',
        NEW1,
        code
    )

# 1b: Read-only filesystem fix for PLY
OLD2 = '''        storePly(ply_path, xyz, rgb)
    try:
        pcd = fetchPly(ply_path)'''
OLD2 = OLD2.replace('\r\n', '\n')

NEW2 = '''        try:
            storePly(ply_path, xyz, rgb)
        except OSError:
            import tempfile, hashlib
            _h = hashlib.md5(path.encode()).hexdigest()[:8]
            ply_path = f"/tmp/pts3d_{_h}.ply"
            storePly(ply_path, xyz, rgb)
    try:
        pcd = fetchPly(ply_path)'''
NEW2 = NEW2.replace('\r\n', '\n')

if OLD2 in code:
    code = code.replace(OLD2, NEW2)
    print('Patch 1b applied: read-only filesystem fix')
else:
    print('Patch 1b: pattern not found!')

# 1c: Optimize storePly memory usage
OLD1c = '''    normals = np.zeros_like(xyz)

    elements = np.empty(xyz.shape[0], dtype=dtype)
    attributes = np.concatenate((xyz, normals, rgb), axis=1)
    elements[:] = list(map(tuple, attributes))'''
OLD1c = OLD1c.replace('\r\n', '\n')

NEW1c = '''    elements = np.empty(xyz.shape[0], dtype=dtype)
    elements['x'] = xyz[:, 0]
    elements['y'] = xyz[:, 1]
    elements['z'] = xyz[:, 2]
    elements['nx'] = 0.0
    elements['ny'] = 0.0
    elements['nz'] = 0.0
    elements['red'] = rgb[:, 0]
    elements['green'] = rgb[:, 1]
    elements['blue'] = rgb[:, 2]'''
NEW1c = NEW1c.replace('\r\n', '\n')

if OLD1c in code:
    code = code.replace(OLD1c, NEW1c)
    print('Patch 1c applied: storePly memory optimization')
else:
    print('Patch 1c: pattern not found!')

with open(dr_path, 'w', newline='\n') as f:
    f.write(code)

# == Patch 2: camera_utils.py ==
cu_path = f'{GS_DIR}/utils/camera_utils.py'
with open(cu_path) as f:
    cu = f.read().replace('\r\n', '\n')

# 2a: Skip missing image files
OLD3 = '''    image = Image.open(cam_info.image_path)'''
OLD3 = OLD3.replace('\r\n', '\n')

NEW3 = '''    if not os.path.exists(cam_info.image_path):
        return None
    image = Image.open(cam_info.image_path)'''
NEW3 = NEW3.replace('\r\n', '\n')

if OLD3 in cu:
    cu = cu.replace(OLD3, NEW3)
    print('Patch 2a applied: skip missing images (loadCam)')
else:
    print('Patch 2a: pattern not found!')

if 'import os' not in cu.split('def ')[0]:
    cu = 'import os\n' + cu

# 2b: Filter None cameras
OLD4 = '''camera_list.append(loadCam(args, id, c, resolution_scale, is_nerf_synthetic, is_test_dataset))'''
OLD4 = OLD4.replace('\r\n', '\n')

NEW4 = '''_c = loadCam(args, id, c, resolution_scale, is_nerf_synthetic, is_test_dataset)
        if _c is not None:
            camera_list.append(_c)'''
NEW4 = NEW4.replace('\r\n', '\n')

if OLD4 in cu:
    cu = cu.replace(OLD4, NEW4)
    print('Patch 2b applied: filter None cameras (cameraList_from_camInfos)')
else:
    OLD4_alt = '''camera_list.append(loadCam(args, id, c, resolution_scale))'''
    OLD4_alt = OLD4_alt.replace('\r\n', '\n')
    
    NEW4_alt = '''_c = loadCam(args, id, c, resolution_scale)
        if _c is not None:
            camera_list.append(_c)'''
    NEW4_alt = NEW4_alt.replace('\r\n', '\n')
    
    if OLD4_alt in cu:
        cu = cu.replace(OLD4_alt, NEW4_alt)
        print('Patch 2b (alt) applied: filter None cameras (cameraList_from_camInfos)')
    else:
        print('Patch 2b: pattern not found!')

with open(cu_path, 'w', newline='\n') as f:
    f.write(cu)

# == Patch 3: gaussian_model.py ==
gm_path = f'{GS_DIR}/scene/gaussian_model.py'
with open(gm_path) as f:
    gm = f.read().replace('\r\n', '\n')

# 3a: Optimize save_ply memory usage
OLD5 = '''        elements = np.empty(xyz.shape[0], dtype=dtype_full)
        attributes = np.concatenate((xyz, normals, f_dc, f_rest, opacities, scale, rotation), axis=1)
        elements[:] = list(map(tuple, attributes))'''
OLD5 = OLD5.replace('\r\n', '\n')

NEW5 = '''        elements = np.empty(xyz.shape[0], dtype=dtype_full)
        elements['x'] = xyz[:, 0]
        elements['y'] = xyz[:, 1]
        elements['z'] = xyz[:, 2]
        elements['nx'] = normals[:, 0]
        elements['ny'] = normals[:, 1]
        elements['nz'] = normals[:, 2]
        for i in range(f_dc.shape[1]):
            elements[f'f_dc_{i}'] = f_dc[:, i]
        for i in range(f_rest.shape[1]):
            elements[f'f_rest_{i}'] = f_rest[:, i]
        elements['opacity'] = opacities[:, 0]
        for i in range(scale.shape[1]):
            elements[f'scale_{i}'] = scale[:, i]
        for i in range(rotation.shape[1]):
            elements[f'rot_{i}'] = rotation[:, i]'''
NEW5 = NEW5.replace('\r\n', '\n')

if OLD5 in gm:
    gm = gm.replace(OLD5, NEW5)
    print('Patch 3 applied: save_ply memory optimization')
else:
    print('Patch 3: pattern not found!')

with open(gm_path, 'w', newline='\n') as f:
    f.write(gm)

print()
print('All patches done!')
print('Data structure per scene:')
print('  images.bin references: ~371 images (original full set)')
print('  train/images/: 240 files  <- used for 3DGS training')
print('  test/images/:   60 files  <- in separate folder, will be skipped')
print('  Missing:        71 files  <- skipped automatically')


In [ ]:
# ============================================================
# CELL 3: Cấu hình Paths & Scene List
# ============================================================
import os

# ---- PATHS ----
# Thay 'vai-nvs-phase1' nếu bạn đặt tên dataset khác trên Kaggle
DATA_ROOT  = '/kaggle/input/vai-nvs-phase1/phase1'
GS_DIR     = '/kaggle/working/gaussian-splatting'
OUT_DIR    = '/kaggle/working/outputs'    # 3DGS checkpoints (sẽ xóa sau render)
RENDER_DIR = '/kaggle/working/renders'   # Ảnh render cuối cùng

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(RENDER_DIR, exist_ok=True)

# ---- SCENES ----
PUBLIC_SCENES  = ['HCM0181', 'HCM0193', 'HCM0204', 'hcm0031', 'hcm0034']
PRIVATE_SCENES = ['HCM0249', 'HCM0254', 'HCM0276', 'HCM1439',
                  'HNI0131', 'HNI0265', 'HNI0366', 'HNI0437']

ALL_SCENES = (
    [('public_set',  s) for s in PUBLIC_SCENES] +
    [('private_set1', s) for s in PRIVATE_SCENES]
)

# ---- HYPERPARAMS ----
TRAIN_ITERATIONS       = 15000
SH_DEGREE              = 3
DENSIFY_UNTIL_ITER     = 10000
LAMBDA_DSSIM           = 0.2
DELETE_CKPT_AFTER_RENDER = True  # Xóa checkpoint để tiết kiệm disk

print(f'Data root : {DATA_ROOT}')
print(f'Total scenes: {len(ALL_SCENES)}')
for ds, sc in ALL_SCENES:
    src = f'{DATA_ROOT}/{ds}/{sc}'
    status = '✅' if os.path.exists(src) else '❌ NOT FOUND'
    print(f'  [{ds}] {sc} — {status}')

In [ ]:
# ============================================================
# CELL 4: Train Loop — Parallel 3D Gaussian Splatting (2 GPUs)
# ============================================================
import subprocess, time, os
from concurrent.futures import ThreadPoolExecutor, as_completed

session_start = time.time()
train_log = []

def train_scene(args):
    gpu_id, dataset, scene = args
    src_path   = f'{DATA_ROOT}/{dataset}/{scene}/train'
    model_path = f'{OUT_DIR}/{dataset}/{scene}'
    ply_path   = f'{model_path}/point_cloud/iteration_{TRAIN_ITERATIONS}/point_cloud.ply'

    if os.path.exists(ply_path):
        print(f'[GPU{gpu_id}] ⏭️  {scene} checkpoint found, skipping.')
        return scene, 0, 'skipped'

    os.makedirs(model_path, exist_ok=True)
    
    # Assign specific GPU to this subprocess via environment variable
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = str(gpu_id)
    env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

    t0 = time.time()
    result = subprocess.run([
        'python', f'{GS_DIR}/train.py',
        '-s', src_path,
        '-m', model_path,
        '--iterations',         str(TRAIN_ITERATIONS),
        '--sh_degree',          str(SH_DEGREE),
        '--densify_until_iter', str(DENSIFY_UNTIL_ITER),
        '--lambda_dssim',       str(LAMBDA_DSSIM),
        '--save_iterations',    str(TRAIN_ITERATIONS),
        '--test_iterations',    '-1',
        '--resolution',         '2',
        '--quiet',
        '--disable_viewer',
        '--data_device',        'cpu'
    ], capture_output=True, text=True, env=env)

    elapsed_min = (time.time() - t0) / 60
    
    if result.returncode != 0:
        print(f'[GPU{gpu_id}] ❌ {scene} FAILED after {elapsed_min:.1f} min')
        print(f'[GPU{gpu_id}] STDERR (last 1500 chars):')
        print(result.stderr[-1500:])
        return scene, elapsed_min, 'failed'
    else:
        print(f'[GPU{gpu_id}] ✅ {scene} done in {elapsed_min:.1f} min')
        return scene, elapsed_min, 'ok'

# Allocate tasks alternately: GPU 0, GPU 1, GPU 0, GPU 1, etc.
tasks = [(i % 2, ds, sc) for i, (ds, sc) in enumerate(ALL_SCENES)]

print(f'Running {len(tasks)} scenes on 2 GPUs in parallel pairs...')
print('Estimated total time: ~3.5 - 4 hours\n')

# Process in batches of 2 tasks (since we have 2 GPUs)
for idx in range(0, len(tasks), 2):
    batch = tasks[idx : idx + 2]
    batch_str = " | ".join([f"GPU{t[0]}:{t[2]}" for t in batch])
    elapsed_h = (time.time() - session_start) / 3600
    print(f'--- Batch {idx//2 + 1}: {batch_str} [Session elapsed: {elapsed_h:.1f}h/12h] ---')
    
    with ThreadPoolExecutor(max_workers=2) as executor:
        futures = {executor.submit(train_scene, t): t for t in batch}
        for future in as_completed(futures):
            sc, elapsed, status = future.result()
            train_log.append((sc, elapsed, status))

print('\n' + '='*60)
print('TRAIN SUMMARY:')
for sc, t, s in train_log:
    print(f'  {sc:15s} — {s:8s} — {t:.1f} min')


In [ ]:
# ============================================================
# CELL 5: Render từ test_poses.csv
# ============================================================
import sys
sys.path.insert(0, GS_DIR)

import torch
import numpy as np
import pandas as pd
from PIL import Image
import os, shutil, time

from gaussian_renderer import render
from scene import GaussianModel

# ---- Helpers ----

def qvec2rotmat(qw, qx, qy, qz):
    """COLMAP quaternion → 3x3 world-to-camera rotation matrix"""
    return np.array([
        [1 - 2*(qy**2 + qz**2),     2*(qx*qy - qz*qw),     2*(qx*qz + qy*qw)],
        [    2*(qx*qy + qz*qw), 1 - 2*(qx**2 + qz**2),     2*(qy*qz - qx*qw)],
        [    2*(qx*qz - qy*qw),     2*(qy*qz + qx*qw), 1 - 2*(qx**2 + qy**2)],
    ])


class SimplePipeline:
    convert_SHs_python  = False
    compute_cov3D_python = False
    debug               = False
    antialiasing         = False


def render_scene(model_path, poses_csv, output_dir, iteration=30000):
    from scene.cameras import Camera

    os.makedirs(output_dir, exist_ok=True)

    ply_path = os.path.join(model_path, f'point_cloud/iteration_{iteration}/point_cloud.ply')
    if not os.path.exists(ply_path):
        print(f'  ❌ PLY not found: {ply_path}')
        return False

    # Load model
    gaussians = GaussianModel(sh_degree=SH_DEGREE)
    gaussians.load_ply(ply_path)

    bg     = torch.tensor([0.0, 0.0, 0.0], dtype=torch.float32, device='cuda')
    pipe   = SimplePipeline()
    df     = pd.read_csv(poses_csv)

    for idx, row in df.iterrows():
        W, H = int(row['width']), int(row['height'])

        R_w2c = qvec2rotmat(row['qw'], row['qx'], row['qy'], row['qz'])
        R     = np.transpose(R_w2c)   # gaussian-splatting Camera expects c2w rotation
        T     = np.array([row['tx'], row['ty'], row['tz']])

        FoVx  = 2 * np.arctan(W / (2 * row['fx']))
        FoVy  = 2 * np.arctan(H / (2 * row['fy']))

        # Tao anh PIL dummy de Camera tu xu ly alpha mask noi bo
        dummy_pil = Image.fromarray(np.zeros((H, W, 3), dtype=np.uint8))

        cam = Camera(
            resolution=(W, H),         # API moi: kich thuoc target (width, height)
            colmap_id=idx, R=R, T=T,
            FoVx=FoVx, FoVy=FoVy,
            depth_params=None,          # API moi: khong co depth
            image=dummy_pil,            # API moi: PIL Image (khong phai tensor)
            invdepthmap=None,           # API moi: khong co inverse depth
            image_name=row['image_name'],
            uid=idx,
            data_device='cuda',
            train_test_exp=False,       # API moi
            is_test_dataset=False,      # API moi
            is_test_view=False          # API moi
        )

        with torch.no_grad():
            pkg = render(cam, gaussians, pipe, bg)

        img_tensor = pkg['render'].clamp(0, 1)
        img_np     = (img_tensor.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
        img_pil    = Image.fromarray(img_np)

        # Resize nếu kích thước khác (edge case)
        if img_pil.size != (W, H):
            img_pil = img_pil.resize((W, H), Image.LANCZOS)

        # Giữ đúng tên file gốc từ image_name (thường là .JPG từ DJI drone)
        out_name = row['image_name']  # Giữ nguyên tên + extension từ CSV
        out_ext  = os.path.splitext(out_name)[1].upper()
        out_path = os.path.join(output_dir, out_name)
        if out_ext in ('.JPG', '.JPEG'):
            img_pil.save(out_path, format='JPEG', quality=92, subsampling=0)
        else:
            img_pil.save(out_path, format='PNG', optimize=True, compress_level=9)

        if idx % 15 == 0:
            print(f'    [{idx+1}/{len(df)}] {row["image_name"]}')

    print(f'  ✅ Rendered {len(df)} images → {output_dir}')
    return True


# ---- Run ----
render_log = []

for dataset, scene in ALL_SCENES:
    model_path = f'{OUT_DIR}/{dataset}/{scene}'
    poses_csv  = f'{DATA_ROOT}/{dataset}/{scene}/test/test_poses.csv'
    output_dir = f'{RENDER_DIR}/{scene}'

    print(f'\n--- Rendering: {scene} ---')
    t0 = time.time()

    ok = render_scene(model_path, poses_csv, output_dir, iteration=TRAIN_ITERATIONS)
    elapsed = (time.time() - t0) / 60
    render_log.append((scene, elapsed, 'ok' if ok else 'failed'))

    # Xóa checkpoint để giải phóng disk
    if ok and DELETE_CKPT_AFTER_RENDER:
        ckpt = f'{model_path}/point_cloud'
        if os.path.exists(ckpt):
            shutil.rmtree(ckpt)
            print(f'  🗑️  Deleted checkpoint to free disk.')

    # Disk status
    _, _, free = shutil.disk_usage('/kaggle/working')
    print(f'  Disk free: {free/1e9:.1f} GB')

print('\nRENDER SUMMARY:')
for sc, t, s in render_log:
    print(f'  {sc:15s} — {s:8s} — {t:.1f} min')

In [ ]:
# ============================================================
# CELL 6: Evaluate Offline trên public_set
# (public_set có ground-truth trong test/images/)
# ============================================================
import torch, lpips, numpy as np
from PIL import Image
from skimage.metrics import structural_similarity as calc_ssim
from skimage.metrics import peak_signal_noise_ratio as calc_psnr
import os, glob

loss_fn = lpips.LPIPS(net='alex').cuda()

PSNR_MAX = 40.0   # Theo đề bài (điều chỉnh nếu BTC thông báo)

def to_tensor(img_np):
    t = torch.from_numpy(img_np).float() / 255.0
    return t.permute(2, 0, 1).unsqueeze(0).cuda() * 2 - 1  # [-1,1] for LPIPS

all_scores = []

for scene in PUBLIC_SCENES:
    gt_dir   = f'{DATA_ROOT}/public_set/{scene}/test/images'
    pred_dir = f'{RENDER_DIR}/{scene}'

    if not os.path.exists(gt_dir) or not os.path.exists(pred_dir):
        print(f'  ⚠️  Skipping {scene} — gt or pred dir missing')
        continue

    lpips_vals, ssim_vals, psnr_vals = [], [], []

    gt_files = sorted(glob.glob(f'{gt_dir}/*.JPG') + glob.glob(f'{gt_dir}/*.jpg') +
                      glob.glob(f'{gt_dir}/*.png'))

    for gt_path in gt_files:
        img_name  = os.path.splitext(os.path.basename(gt_path))[0] + '.png'
        pred_path = os.path.join(pred_dir, img_name)
        if not os.path.exists(pred_path):
            # Try same extension
            pred_path = os.path.join(pred_dir, os.path.basename(gt_path))
        if not os.path.exists(pred_path):
            continue

        gt_np   = np.array(Image.open(gt_path).convert('RGB'))
        pred_np = np.array(Image.open(pred_path).convert('RGB'))

        # Resize pred → match gt nếu cần
        if gt_np.shape != pred_np.shape:
            pred_np = np.array(Image.fromarray(pred_np).resize(
                (gt_np.shape[1], gt_np.shape[0]), Image.LANCZOS))

        # LPIPS
        with torch.no_grad():
            lp = loss_fn(to_tensor(gt_np), to_tensor(pred_np)).item()
        lpips_vals.append(lp)

        # SSIM
        ss = calc_ssim(gt_np, pred_np, channel_axis=2, data_range=255)
        ssim_vals.append(ss)

        # PSNR
        ps = calc_psnr(gt_np, pred_np, data_range=255)
        psnr_vals.append(ps)

    if not lpips_vals:
        print(f'  ⚠️  No matched images for {scene}')
        continue

    avg_lpips = np.mean(lpips_vals)
    avg_ssim  = np.mean(ssim_vals)
    avg_psnr  = np.mean(psnr_vals)
    psnr_norm = min(avg_psnr / PSNR_MAX, 1.0)
    score     = 0.4 * (1 - avg_lpips) + 0.3 * avg_ssim + 0.3 * psnr_norm

    all_scores.append(score)
    print(f'{scene:15s} | LPIPS={avg_lpips:.4f} | SSIM={avg_ssim:.4f} | '
          f'PSNR={avg_psnr:.2f}dB | Score={score:.4f}')

if all_scores:
    print(f'\n  ★ Mean Score (public_set): {np.mean(all_scores):.4f}')

In [ ]:
# ============================================================
# CELL 7: Validate renders & Package submission.zip
# ============================================================
import zipfile, glob, os, pandas as pd

ZIP_PATH = '/kaggle/working/submission_round1.zip'
errors   = []

print('=== Validating renders ===')
for dataset, scene in ALL_SCENES:
    poses_csv  = f'{DATA_ROOT}/{dataset}/{scene}/test/test_poses.csv'
    render_dir = f'{RENDER_DIR}/{scene}'

    df = pd.read_csv(poses_csv)
    expected = len(df)

    if not os.path.exists(render_dir):
        errors.append(f'❌ {scene}: render dir missing')
        continue

    expected_names = set(df['image_name'].tolist())
    found_files = set(os.path.basename(p) for p in
        glob.glob(f'{render_dir}/*.png') +
        glob.glob(f'{render_dir}/*.PNG') +
        glob.glob(f'{render_dir}/*.jpg') +
        glob.glob(f'{render_dir}/*.JPG'))
    found = len(found_files)
    missing = expected_names - found_files
    status = '✅' if not missing else f'⚠️  {found}/{expected}'
    print(f'  {scene:15s}: {status}')
    if missing:
        sample = list(missing)[:5]
        print(f'    Missing ({len(missing)}): {sample}' + ('...' if len(missing)>5 else ''))
        errors.append(f'{scene}: missing {len(missing)} images')

if errors:
    print('\nErrors:')
    for e in errors:
        print(f'  {e}')

print('\n=== Creating submission_round1.zip ===')
# Use ZIP_STORED for JPEG (already compressed) — DEFLATE saves <1%
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_STORED) as zf:
    for dataset, scene in ALL_SCENES:
        render_dir = f'{RENDER_DIR}/{scene}'
        if not os.path.exists(render_dir):
            print(f'  ⚠️ {scene}: render dir not found, skipping')
            continue
        imgs = sorted(
            glob.glob(f'{render_dir}/*.png') +
            glob.glob(f'{render_dir}/*.PNG') +
            glob.glob(f'{render_dir}/*.jpg') +
            glob.glob(f'{render_dir}/*.JPG')
        )
        for img_path in imgs:
            arcname = f'{scene}/{os.path.basename(img_path)}'
            zf.write(img_path, arcname)
        print(f'  {scene}: {len(imgs)} images')

size_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f'\n✅ submission_round1.zip created: {size_mb:.1f} MB')
print(f'   Path: {ZIP_PATH}')
if size_mb > 500:
    print(f'   ⚠️  WARNING: {size_mb:.1f} MB > 500 MB limit! Consider lowering JPEG quality.')
else:
    print(f'   ✅ Size OK ({size_mb:.1f} MB < 500 MB limit)')
print('   → Download từ Kaggle Output panel rồi submit!')

In [ ]:
# ============================================================
# CELL 8 (Optional): Quick visual check — xem vài ảnh render
# ============================================================
import matplotlib.pyplot as plt
from PIL import Image
import glob, os

scene    = PUBLIC_SCENES[0]   # Đổi scene nếu muốn
imgs     = sorted(glob.glob(f'{RENDER_DIR}/{scene}/*.png'))[:4]

fig, axes = plt.subplots(1, len(imgs), figsize=(16, 4))
fig.suptitle(f'Render preview — {scene}', fontsize=14)
for ax, p in zip(axes, imgs):
    ax.imshow(Image.open(p))
    ax.set_title(os.path.basename(p), fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()